## 1. Environment setup

In [2]:
import os
import sys

try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    print("Running in Colab.")
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    ROOT_DIR = "/content/drive/MyDrive/omg_diploma_2025/restore_punct"
    if ROOT_DIR not in sys.path:
        sys.path.append(ROOT_DIR)
else:
    print("Running locally.")
    ROOT_DIR = os.getcwd()

DATA_DIR = os.path.join(ROOT_DIR, "data")
os.makedirs(DATA_DIR, exist_ok=True)
print("DATA_DIR =", DATA_DIR)


Running locally.
DATA_DIR = /home/temari/god please no diploma/restore_punct/data


In [3]:
if IS_COLAB:
    !pip install -q datasets pandas conllu tqdm nltk openpyxl requests

import ast
import io
import gc
import json
import random
import re
import zipfile
from collections import Counter

import conllu
import nltk
import numpy as np
import pandas as pd
import requests
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

try:
    from nltk.tokenize import sent_tokenize
    sent_tokenize("тест.", language='russian')
except LookupError:
    nltk.download('punkt')
    nltk.download('punkt_tab')
    from nltk.tokenize import sent_tokenize

from extract_labels_standardized import (
    LABEL_MAP,
    ID_TO_LABEL,
    extract_labels_standardized,
    robust_clean_text,
)


## 2. Configuration

In [4]:
MAX_WORDS_PER_CHUNK = 250

TARGET_GAZETA = 15_000
TARGET_OPUS_CHUNKS = 60_000
TARGET_WIKI_CHUNKS = 15_000

VAL_FRACTION = 0.05
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


## 3. Label map

In [5]:
print(f"{len(LABEL_MAP)} classes:")
for label, idx in LABEL_MAP.items():
    print(f"  {idx:2d}  {label!r}")

label_map_path = os.path.join(DATA_DIR, "label_map.json")
with open(label_map_path, 'w', encoding='utf-8') as f:
    json.dump(ID_TO_LABEL, f, ensure_ascii=False)
print(f"\nWrote {label_map_path}")


28 classes:
   0  'O'
   1  '.'
   2  ','
   3  '!'
   4  '?'
   5  ':'
   6  ';'
   7  '-'
   8  '"'
   9  ', "'
  10  ': "'
  11  '. "'
  12  '"?'
  13  '"!'
  14  '",'
  15  '".'
  16  '?"'
  17  '!"'
  18  '" -'
  19  '- "'
  20  '", -'
  21  '!" -'
  22  '?" -'
  23  '. -'
  24  '""'
  25  '! -'
  26  '? -'
  27  ', -'

Wrote /home/temari/god please no diploma/restore_punct/data/label_map.json


## 4. Utilities

In [ ]:
def create_json_dataset(texts, output_filename):
    """
    Each produced example is ``{'tokens': [...], 'ner_tags': [...]}``.
    """
    processed = []
    print(f"Labeling {len(texts)} texts -> {output_filename}")
    for text in tqdm(texts):
        try:
            pairs = extract_labels_standardized(text)
        except Exception:
            continue
        if len(pairs) < 5:
            continue
        processed.append({
            "tokens":   [p['word']  for p in pairs],
            "ner_tags": [p['label'] for p in pairs],
        })

    full_path = os.path.join(DATA_DIR, output_filename)
    with open(full_path, 'w', encoding='utf-8') as f:
        json.dump(processed, f, ensure_ascii=False)
    print(f"Saved {len(processed)} examples to {full_path}")
    del processed
    gc.collect()


In [ ]:
def _hard_split(words, max_words): # helper
    if len(words) <= max_words:
        return [words]
    return [words[i:i + max_words] for i in range(0, len(words), max_words)]


def pack_sentences(sentences, max_words=MAX_WORDS_PER_CHUNK):
    """
    pack sentences into chunks up to ``max_words`` words.
    """
    chunks, buf, buf_n = [], [], 0
    for sent in sentences:
        words = sent.split()
        if not words:
            continue
        for piece in _hard_split(words, max_words):
            n = len(piece)
            if buf and buf_n + n > max_words:
                chunks.append(" ".join(buf))
                buf, buf_n = [], 0
            buf.append(" ".join(piece))
            buf_n += n
    if buf:
        chunks.append(" ".join(buf))
    return chunks


def chunk_external_text(text, max_words=MAX_WORDS_PER_CHUNK):
    return pack_sentences(sent_tokenize(text, language='russian'), max_words)


## 5. SynTagRus

In [ ]:
SYNTAGRUS_TRAIN_URLS = [
    "https://raw.githubusercontent.com/UniversalDependencies/UD_Russian-SynTagRus/master/ru_syntagrus-ud-train-a.conllu",
    "https://raw.githubusercontent.com/UniversalDependencies/UD_Russian-SynTagRus/master/ru_syntagrus-ud-train-b.conllu",
    "https://raw.githubusercontent.com/UniversalDependencies/UD_Russian-SynTagRus/master/ru_syntagrus-ud-train-c.conllu",
]
SYNTAGRUS_TEST_URL = "https://raw.githubusercontent.com/UniversalDependencies/UD_Russian-SynTagRus/master/ru_syntagrus-ud-test.conllu"


def process_syntagrus(urls, name):
    chunks = []
    print(f"--- Processing SynTagRus [{name}] ---")
    for url in urls:
        fname = url.rsplit("/", 1)[-1]
        file_path = os.path.join(DATA_DIR, fname)
        if not os.path.exists(file_path):
            r = requests.get(url, timeout=120)
            r.raise_for_status()
            with open(file_path, "wb") as f:
                f.write(r.content)

        doc_sents, cur_doc = [], None
        with open(file_path, "r", encoding="utf-8") as f:
            for sent in tqdm(conllu.parse_incr(f), desc=fname):
                sent_id = sent.metadata['sent_id']
                text = sent.metadata['text']
                doc_id = sent_id.rsplit("_", 1)[0] if "_" in sent_id else sent_id

                if doc_id != cur_doc:
                    if doc_sents:
                        chunks.extend(pack_sentences(doc_sents, MAX_WORDS_PER_CHUNK))
                    doc_sents, cur_doc = [], doc_id
                doc_sents.append(text)

        if doc_sents:
            chunks.extend(pack_sentences(doc_sents, MAX_WORDS_PER_CHUNK))
    print(f"{name}: {len(chunks)} chunks.")
    return chunks


In [8]:
syn_train = process_syntagrus(SYNTAGRUS_TRAIN_URLS, "train")
syn_test  = process_syntagrus([SYNTAGRUS_TEST_URL],    "test")


--- Processing SynTagRus [train] ---


ru_syntagrus-ud-train-a.conllu: 0it [00:00, ?it/s]

ru_syntagrus-ud-train-b.conllu: 0it [00:00, ?it/s]

ru_syntagrus-ud-train-c.conllu: 0it [00:00, ?it/s]

train: 4418 chunks.
--- Processing SynTagRus [test] ---


ru_syntagrus-ud-test.conllu: 0it [00:00, ?it/s]

test: 569 chunks.


## 6. Gazeta (journalism)

Long-form news articles. Streamed from HuggingFace and sentence-packed into ``MAX_WORDS_PER_CHUNK``-word chunks.

In [9]:
ext_journalism = []
ds_gazeta = load_dataset("IlyaGusev/gazeta", split="train", streaming=True)

count = 0
for item in tqdm(ds_gazeta, desc="Gazeta stream"):
    if count >= TARGET_GAZETA:
        break
    t = item['text'].replace('\n', ' ')
    if len(t) <= 100:
        continue
    ext_journalism.extend(chunk_external_text(t))
    count += 1

print(f"Gazeta: {len(ext_journalism)} chunks from {count} articles.")


Gazeta stream: 0it [00:00, ?it/s]

Gazeta: 43513 chunks from 15000 articles.


## 7. OPUS Books (fiction)

Literary prose with heavy direct-speech and quotation patterns — important for the rare quote-dash classes (``!" -``, ``?" -``, ``", -`` etc.). The archive contains one sentence per line, so consecutive lines are grouped into paragraph-sized buffers before chunking.

In [ ]:
ext_fiction = []
OPUS_URL = "https://object.pouta.csc.fi/OPUS-Books/v1/moses/en-ru.txt.zip"

resp = requests.get(OPUS_URL, stream=True, timeout=300)
resp.raise_for_status()

BUFFER_LINES = 20  # group N sentences before chunking 

with io.BytesIO() as buf:
    for chunk in resp.iter_content(chunk_size=1 << 14):
        buf.write(chunk)
    print("Unzipping OPUS...")
    with zipfile.ZipFile(buf) as z:
        with z.open('Books.en-ru.ru') as f:
            line_buf = []
            for line in tqdm(f, desc="OPUS lines"):
                if len(ext_fiction) >= TARGET_OPUS_CHUNKS:
                    break
                try:
                    t = line.decode('utf-8').strip()
                except UnicodeDecodeError:
                    continue
                if len(t) <= 20:
                    continue
                line_buf.append(t)
                if len(line_buf) >= BUFFER_LINES:
                    ext_fiction.extend(chunk_external_text(" ".join(line_buf)))
                    line_buf = []
            if line_buf:
                ext_fiction.extend(chunk_external_text(" ".join(line_buf)))

ext_fiction = ext_fiction[:TARGET_OPUS_CHUNKS]
print(f"OPUS Books: {len(ext_fiction)} chunks.")


Unzipping OPUS...


OPUS lines: 0it [00:00, ?it/s]

OPUS Books: 1545 chunks.


## 8. Wikipedia (encyclopedic / nonfiction)

Streamed from ``wiki40b`` (Russian). Each raw article has ``_START_ARTICLE_`` / ``_START_SECTION_`` / ``_START_PARAGRAPH_`` tags; these are used to split into segments and drop unpunctuated headings (which would otherwise teach the model to emit no punctuation for arbitrary noun phrases).

In [ ]:
def check_heading_punctuation(heading_text):
    if not heading_text:
        return False
    pairs = extract_labels_standardized(heading_text)
    if not pairs:
        return False
    return pairs[-1]['label'] != LABEL_MAP['O']


_WIKI_TAGS = {
    "_START_ARTICLE_":   "article_heading",
    "_START_SECTION_":   "section_heading",
    "_START_PARAGRAPH_": "paragraph",
}
_WIKI_TAG_RE = "|".join(re.escape(t) for t in _WIKI_TAGS)


def parse_wiki_segments(raw_item_text):
    if isinstance(raw_item_text, bytes):
        text = raw_item_text.decode('utf-8', errors='replace')
    else:
        text = raw_item_text
    if text.startswith(("b'", 'b"')):
        try:
            text = ast.literal_eval(text).decode('utf-8')
        except (ValueError, SyntaxError):
            pass
    text = text.replace('_NEWLINE_', ' ')
    text = re.sub(r'\s+', ' ', text).strip()

    segments = []
    matches = list(re.finditer(_WIKI_TAG_RE, text))
    if not matches:
        cleaned = robust_clean_text(text)
        return [{"text": cleaned, "type": "paragraph"}] if cleaned else []

    if matches[0].start() > 0:
        pre = robust_clean_text(text[:matches[0].start()])
        if pre:
            segments.append({"text": pre, "type": "paragraph"})

    for i, m in enumerate(matches):
        tag = m.group(0)
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        seg_text = robust_clean_text(text[start:end])
        if not seg_text:
            continue
        seg_type = _WIKI_TAGS[tag]
        if seg_type in ("article_heading", "section_heading") \
                and not check_heading_punctuation(seg_text):
            continue
        segments.append({"text": seg_text, "type": seg_type})

    return segments


In [12]:
ext_science = []
ds_wiki = load_dataset("wiki40b", "ru", split="train", streaming=True)

for item in tqdm(ds_wiki, desc="Wiki stream"):
    if len(ext_science) >= TARGET_WIKI_CHUNKS:
        break
    for seg in parse_wiki_segments(item['text']):
        if len(seg['text']) <= 100:
            continue
        ext_science.extend(chunk_external_text(seg['text']))
        if len(ext_science) >= TARGET_WIKI_CHUNKS:
            ext_science = ext_science[:TARGET_WIKI_CHUNKS]
            break

print(f"Wiki40b: {len(ext_science)} chunks.")


Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Wiki stream: 0it [00:00, ?it/s]

Wiki40b: 15000 segments.


## 9. Assemble the mixed train/val/benchmark triple

Mix SynTagRus-train with the external sources, shuffle deterministically, carve off ``VAL_FRACTION`` as internal validation, and label the rest. SynTagRus *test* becomes the held-out general benchmark.

In [ ]:
_LIBRUSEC_FILES = [
    f"hf://datasets/IlyaGusev/librusec/{i:02d}.jsonl.zst" for i in range(20)
]
ds_probe = load_dataset("json", data_files=_LIBRUSEC_FILES,
                        split="train", streaming=True)

total_texts = 0
texts_with_q = 0
total_q_chars = 0
sample_contexts = []

for item in ds_probe:
    text = item.get("text", "")
    if not isinstance(text, str):
        continue
    total_texts += 1
    count = text.count("?")
    if count:
        texts_with_q += 1
        total_q_chars += count
        if len(sample_contexts) < 5:
            idx = text.index("?")
            lo, hi = max(0, idx - 40), min(len(text), idx + 40)
            sample_contexts.append(text[lo:hi])
    if total_texts >= 5000:
        break

print(f"LibruSec probe ({total_texts} texts):")
print(f"  Texts with '?'  : {texts_with_q}")
print(f"  Total '?' chars : {total_q_chars}")
print(f"\n  Sample contexts around '?':")
for i, ctx in enumerate(sample_contexts, 1):
    print(f"    {i}. ...{ctx!r}...")

In [13]:
full_corpus = syn_train + ext_journalism + ext_fiction + ext_science
print("Source sizes:")
print(f"  SynTagRus-train  : {len(syn_train):>7d}")
print(f"  Gazeta           : {len(ext_journalism):>7d}")
print(f"  OPUS Books       : {len(ext_fiction):>7d}")
print(f"  Wiki40b          : {len(ext_science):>7d}")
print(f"  TOTAL            : {len(full_corpus):>7d}")

rng = random.Random(RANDOM_SEED)
rng.shuffle(full_corpus)

train_texts, val_texts = train_test_split(
    full_corpus, test_size=VAL_FRACTION, random_state=RANDOM_SEED,
)
print(f"\nTrain = {len(train_texts)}   Val = {len(val_texts)}")

create_json_dataset(train_texts, "train_all.json")
create_json_dataset(val_texts,   "val_internal.json")
create_json_dataset(syn_test,    "bench_test_all.json")


Source sizes:
  SynTagRus-train  :    4418
  Gazeta           :   43513
  OPUS Books       :    1545
  Wiki40b          :   15000
  TOTAL            :   64476

Train = 61252   Val = 3224
Labeling 61252 texts -> train_all.json


  0%|          | 0/61252 [00:00<?, ?it/s]

Saved 61243 examples to /home/temari/god please no diploma/restore_punct/data/train_all.json
Labeling 3224 texts -> val_internal.json


  0%|          | 0/3224 [00:00<?, ?it/s]

Saved 3224 examples to /home/temari/god please no diploma/restore_punct/data/val_internal.json
Labeling 569 texts -> bench_test_all.json


  0%|          | 0/569 [00:00<?, ?it/s]

Saved 569 examples to /home/temari/god please no diploma/restore_punct/data/bench_test_all.json


## 10. LORuGEC — learner-error benchmark

Ships a single corrected-sentences spreadsheet (``LORuGEC.xlsx``) plus two M2 files that pin the *official* validation and test splits. We recover the splits by matching the ``Initial sentence`` column to the ``S`` lines in the M2 files, then:

- the val-pool is split 85/15 into our ``train_lorugec`` / ``val_lorugec`` (fine-tuning data);
- the test-pool is kept intact as the held-out ``bench_test_lorugec``.


In [ ]:
LORUGEC_XLSX_URL     = "https://raw.githubusercontent.com/ReginaNasyrova/LORuGEC/main/LORuGEC.xlsx"
LORUGEC_VAL_M2_URL   = "https://raw.githubusercontent.com/ReginaNasyrova/LORuGEC/main/LORuGEC.val.m2"
LORUGEC_TEST_M2_URL  = "https://raw.githubusercontent.com/ReginaNasyrova/LORuGEC/main/LORuGEC.test.m2"


def _m2_initial_sentences(url):
    m2 = requests.get(url, timeout=120).text
    return [line[2:].strip() for line in m2.split('\n') if line.startswith('S ')]


resp = requests.get(LORUGEC_XLSX_URL, timeout=120)
resp.raise_for_status()
df_lorugec = pd.read_excel(io.BytesIO(resp.content), engine='openpyxl')
df_lorugec['Initial sentence'] = df_lorugec['Initial sentence'].astype(str).str.strip()
print(f"LORuGEC spreadsheet: {len(df_lorugec)} rows.")

val_initials  = _m2_initial_sentences(LORUGEC_VAL_M2_URL)
test_initials = _m2_initial_sentences(LORUGEC_TEST_M2_URL)

val_pool  = df_lorugec[df_lorugec['Initial sentence'].isin(val_initials)]['Correct sentence']
test_pool = df_lorugec[df_lorugec['Initial sentence'].isin(test_initials)]['Correct sentence']

val_pool_texts  = [robust_clean_text(t) for t in val_pool.dropna().astype(str)]
test_pool_texts = [robust_clean_text(t) for t in test_pool.dropna().astype(str)]
print(f"  val-pool  (becomes train_lorugec) : {len(val_pool_texts)} sentences")
print(f"  test-pool (becomes bench_test)    : {len(test_pool_texts)} sentences")

create_json_dataset(val_pool_texts,  "train_lorugec.json")
create_json_dataset(test_pool_texts, "bench_test_lorugec.json")


LORuGEC spreadsheet: 960 rows.
  val-pool  (becomes train_lorugec) : 348 sentences
  test-pool (becomes bench_test)    : 612 sentences
Labeling 348 texts -> train_lorugec.json


  0%|          | 0/348 [00:00<?, ?it/s]

Saved 336 examples to /home/temari/god please no diploma/restore_punct/data/train_lorugec.json
Labeling 612 texts -> bench_test_lorugec.json


  0%|          | 0/612 [00:00<?, ?it/s]

Saved 603 examples to /home/temari/god please no diploma/restore_punct/data/bench_test_lorugec.json


## 11. GERA — essay-error benchmark

Plain M2 files with multiple annotators; we pick annotator 0 and apply their edits to produce the corrected sentence for each entry.

In [ ]:
GERA_URLS = {
    "train": "https://raw.githubusercontent.com/ReginaNasyrova/GERA/main/GERA.train.m2",
    "val":   "https://raw.githubusercontent.com/ReginaNasyrova/GERA/main/GERA.dev.m2",
    "test":  "https://raw.githubusercontent.com/ReginaNasyrova/GERA/main/GERA.test.m2",
}


def m2_to_corrected_sentences(m2_content, annotator_id=0):
    """Apply annotator-N edits to each ``S``-line to yield corrected strings."""
    out = []
    for block in m2_content.strip().split('\n\n'):
        lines = block.split('\n')
        if not lines or not lines[0].startswith('S '):
            continue
        tokens = lines[0][2:].split()
        edits = []
        for line in lines[1:]:
            if not line.startswith('A '):
                continue
            parts = line[2:].split('|||')
            if len(parts) < 6:
                continue
            span = parts[0].split()
            start, end = int(span[0]), int(span[1])
            err_type, cor = parts[1], parts[2]
            ann = int(parts[-1])
            if ann == annotator_id and err_type != 'noop':
                edits.append((start, end, cor))
        edits.sort(key=lambda e: (e[0], e[1]), reverse=True)
        for start, end, cor in edits:
            cor_tokens = [] if cor == '-NONE-' else (cor.split() if cor else [])
            tokens[start:end] = cor_tokens
        out.append(" ".join(tokens))
    return out

def process_gera(split, url, output_filename):
    print(f"--- GERA {split} ---")
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    sents = m2_to_corrected_sentences(r.text)
    sents = [robust_clean_text(s) for s in sents if s.strip()]
    print(f"  {len(sents)} sentences.")
    create_json_dataset(sents, output_filename)


process_gera("train", GERA_URLS["train"], "train_gera.json")
process_gera("val",   GERA_URLS["val"],   "val_gera.json")
process_gera("test",  GERA_URLS["test"],  "bench_test_gera.json")


--- GERA train ---
  4592 sentences.
Labeling 4592 texts -> train_gera.json


  0%|          | 0/4592 [00:00<?, ?it/s]

Saved 4379 examples to /home/temari/god please no diploma/restore_punct/data/train_gera.json
--- GERA val ---
  775 sentences.
Labeling 775 texts -> val_gera.json


  0%|          | 0/775 [00:00<?, ?it/s]

Saved 747 examples to /home/temari/god please no diploma/restore_punct/data/val_gera.json
--- GERA test ---
  1314 sentences.
Labeling 1314 texts -> bench_test_gera.json


  0%|          | 0/1314 [00:00<?, ?it/s]

Saved 1259 examples to /home/temari/god please no diploma/restore_punct/data/bench_test_gera.json


## 12. Audit & sanity checks

Load every produced file and report:
1. a few sample token/label rows,
2. per-file coverage of all 28 classes — flags any class that ended up with zero support.


In [16]:
def visualize_dataset(examples, num_samples=3, id2label=ID_TO_LABEL):
    rows = list(examples)[:num_samples] if not isinstance(examples, list) else examples[:num_samples]
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    df['ner_tag_names'] = df['ner_tags'].apply(lambda ids: [id2label[int(i)] for i in ids])
    return df[['tokens', 'ner_tag_names', 'ner_tags']]


def audit_example(example, id2label=ID_TO_LABEL):
    rows = [
        {"Token": tok, "Tag ID": int(tid), "Label": id2label[int(tid)]}
        for tok, tid in zip(example['tokens'], example['ner_tags'])
    ]
    return pd.DataFrame(rows)


pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 200)


In [17]:
FILES_TO_AUDIT = [
    "train_all.json", "val_internal.json", "bench_test_all.json",
    "train_lorugec.json", "bench_test_lorugec.json",
    "train_gera.json", "val_gera.json", "bench_test_gera.json",
]

summary_rows = []
for fname in FILES_TO_AUDIT:
    path = os.path.join(DATA_DIR, fname)
    if not os.path.exists(path):
        print(f"MISSING: {fname}")
        continue
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    counter = Counter()
    n_tokens = 0
    max_words = 0
    for ex in data:
        counter.update(ex['ner_tags'])
        n_tokens += len(ex['ner_tags'])
        if len(ex['tokens']) > max_words:
            max_words = len(ex['tokens'])
    seen = {int(k) for k in counter}
    missing = sorted(set(LABEL_MAP.values()) - seen)
    summary_rows.append({
        "file":          fname,
        "examples":      len(data),
        "tokens":        n_tokens,
        "max_words":     max_words,
        "classes_seen":  len(seen),
        "missing_ids":   missing,
        "missing_labels": [ID_TO_LABEL[i] for i in missing],
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
# Safety check: no chunk should exceed MAX_WORDS_PER_CHUNK.
over = summary_df[summary_df["max_words"] > MAX_WORDS_PER_CHUNK]
if len(over):
    print("\nWARNING: chunks exceeding MAX_WORDS_PER_CHUNK found:")
    print(over[["file", "max_words"]].to_string(index=False))
else:
    print(f"\nOK: every chunk is <= {MAX_WORDS_PER_CHUNK} words.")


                   file  examples   tokens  max_words  classes_seen                                     missing_ids                                             missing_labels
         train_all.json     61243 11487730       3479            28                                              []                                                         []
      val_internal.json      3224   607105       1840            28                                              []                                                         []
    bench_test_all.json       569   127657        272            28                                              []                                                         []
     train_lorugec.json       336     4605         36            17     [9, 11, 12, 20, 21, 22, 23, 24, 25, 26, 27]   [, ", . ", "?, ", -, !" -, ?" -, . -, "", ! -, ? -, , -]
bench_test_lorugec.json       603     8464         46            16 [9, 10, 11, 12, 13, 16, 17, 19, 21, 22, 23, 25] [, ", : "

In [ ]:
with open(os.path.join(DATA_DIR, "train_all.json"), 'r', encoding='utf-8') as f:
    train_all = json.load(f)
print(f"train_all.json: {len(train_all)} examples")
display(visualize_dataset(train_all, num_samples=3))
print()
print("Full audit of train_all[0]:")
display(audit_example(train_all[0]).head(40))


train_all.json: 61243 examples


,tokens,ner_tag_names,ner_tags
0,"[Еще, утверждает, что, слышал, выстрелы, с, расстояния, 60, или, 70, метров, хотя, на, оружии, имелся, глушитель, перечисляет, адвокат, Исаева, Муса, Хадисов, Он, напомнил, что, ни, его, подзащитный, ни, другие, фигуранты, дела, не, признают, своей, вины, Только, Хацуев, сознался, в, незаконном, хранении, оружия, Обвиняемые, выступят, после, допроса, всех, свидетелей, и, объяснят, что, не, имеют, к, этим, событиям, никакого, отношения, заявил, Хадисов, Газете, Ru]","[O, ,, O, O, O, O, O, O, O, O, ,, O, O, O, O, "", -, O, O, O, O, ., O, ,, O, O, O, ,, O, O, O, O, O, O, O, ., O, O, O, O, O, O, . "", O, O, O, O, O, O, O, ,, O, O, O, O, O, O, O, "", -, O, "", ., "".]","[0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 20, 0, 0, 0, 0, 1, 0, 2, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 11, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 20, 0, 8, 1, 15]"
1,"[На, процесс, липолиза, оказывают, стимулирующее, воздействие, гормоны, глюкагон, и, соматотропин, Противоположное, действие, оказывает, инсулин, который, стимулирует, фосфодиэстеразу, расщепляющую, цАМФ, молекулу, вторичного, посредника, что, тормозит, процесс, липолиза]","[O, O, O, O, O, O, O, O, O, ., O, O, O, ,, O, O, ,, O, -, O, O, ,, O, O, O, .]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0, 2, 0, 7, 0, 0, 2, 0, 0, 0, 1]"
2,"[Расположено, между, Лесным, озером, и, Великими, озёрами, на, границе, канадской, провинции, Онтарио, и, американского, штата, Миннесота, Одно, из, крупных, озёр, Канады, общая, площадь, равна, 932, км², на, территории, Канады, находится, большая, часть, озера, площадью, 741, км², (площадь, водной, поверхности, 692, км²), Является, остатком, огромного, ледникового, озера, Агассис, Высота, над, уровнем, моря, 338, метров, колебания, уровня, озера, до, 0,14, метра, Ледостав, с, ноября, по, май, Сток, из, озера, по, одноимённой, реке, в, Лесное, озеро, На, берегах, реки, находятся, города, Интернешнл, Фоллс, (США), и, Форт-Франсес, (Канада)]","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ., O, O, O, O, -, O, O, O, O, ,, O, O, O, O, O, O, O, O, O, O, O, O, -, O, ., O, O, O, O, O, ., O, O, O, O, O, ,, O, O, O, O, O, ., O, O, O, O, ., O, O, O, O, O, O, O, O, ., O, O, O, O, O, O, O, O, O, O, .]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 7, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]"



Full audit of train_all[0]:


,Token,Tag ID,Label
0,Еще,0,O
1,утверждает,2,","
2,что,0,O
3,слышал,0,O
4,выстрелы,0,O
5,с,0,O
6,расстояния,0,O
7,60,0,O
8,или,0,O
9,70,0,O


In [19]:
tag_counts = Counter()
for ex in train_all:
    tag_counts.update(ex['ner_tags'])
total = sum(tag_counts.values())
rows = []
for label, idx in LABEL_MAP.items():
    c = tag_counts.get(idx, 0)
    rows.append({"id": idx, "label": label, "count": c, "pct": 100 * c / total if total else 0.0})
dist = pd.DataFrame(rows).sort_values("count", ascending=False)
print(dist.to_string(index=False))


 id label   count       pct
  0     O 9310538 81.047674
  2     ,  906621  7.892081
  1     .  614422  5.348507
  8     "  290361  2.527575
  7     -   83232  0.724530
 15    ".   50652  0.440923
  5     :   46471  0.404527
 11   . "   41766  0.363571
 14    ",   33656  0.292973
 20  ", -   29607  0.257727
 23   . -   17179  0.149542
 27   , -   14879  0.129521
  9   , "    7717  0.067176
  6     ;    6923  0.060264
  4     ?    6767  0.058906
 10   : "    6585  0.057322
 18   " -    4907  0.042715
 26   ? -    4870  0.042393
  3     !    3530  0.030728
 19   - "    3265  0.028422
 24    ""    1067  0.009288
 17    !"     731  0.006363
 16    ?"     526  0.004579
 22  ?" -     445  0.003874
 21  !" -     353  0.003073
 25   ! -     334  0.002907
 12    "?     256  0.002228
 13    "!      70  0.000609


## Fiction rare punctuation mining

In [ ]:
# config
import hashlib
import sqlite3

QUOTE_CHARS = frozenset('"«»\u201c\u201d\u201e\u201f')
DASH_CHARS  = frozenset('-\u2010\u2011\u2012\u2013\u2014\u2015\u2212')

_ABUNDANT = {LABEL_MAP["O"], LABEL_MAP["."], LABEL_MAP[","]}
USEFUL_LABEL_IDS = frozenset(v for v in LABEL_MAP.values() if v not in _ABUNDANT)

def _find_categories(text):
    """Return mining-category keys this text qualifies for."""
    cats = set()
    for ch in ("!", "?", ";"):
        if ch in text:
            cats.add(ch)
    has_quote = any(ch in QUOTE_CHARS for ch in text)
    has_dash  = any(ch in DASH_CHARS  for ch in text)
    if has_quote and has_dash:
        cats.add("dialogue")
    return cats


def _has_useful_labels(pairs):
    return any(p["label"] in USEFUL_LABEL_IDS for p in pairs)


def _hash_chunk(text):
    return hashlib.blake2b(text.encode("utf-8"), digest_size=16).digest()

_CATEGORIES = ("!", "?", ";", "dialogue")
_counts  = {c: 0 for c in _CATEGORIES}
tag_counts = Counter()


def mine_from_texts(texts_iter, source, desc, conn, out_fp):
    state = {
        "written": 0,
        "duplicates": 0,
        "dropped_only_abundant": 0,
        "errors": 0,
    }

    for raw in tqdm(texts_iter, desc=desc):
        if not raw or not isinstance(raw, str) or len(raw) < 50:
            continue

        for para in (p.strip() for p in raw.split("\n") if len(p.strip()) > 30):
            if not _find_categories(para):
                continue

            for chunk in chunk_external_text(para, MAX_WORDS_PER_CHUNK):
                cats = _find_categories(chunk)
                if not cats:
                    continue

                best = min(cats, key=lambda c: _counts[c])
                _counts[best] += 1

                chunk_hash = _hash_chunk(chunk)
                try:
                    conn.execute("INSERT INTO seen_chunks(hash) VALUES (?)", (chunk_hash,))
                except sqlite3.IntegrityError:
                    state["duplicates"] += 1
                    continue

                try:
                    pairs = extract_labels_standardized(chunk)
                except Exception:
                    state["errors"] += 1
                    continue

                if len(pairs) < 5:
                    continue
                if not _has_useful_labels(pairs):
                    state["dropped_only_abundant"] += 1
                    continue

                ex = {
                    "tokens": [p["word"] for p in pairs],
                    "ner_tags": [p["label"] for p in pairs],
                }
                if state["written"]:
                    out_fp.write(",\n")
                json.dump(ex, out_fp, ensure_ascii=False)
                state["written"] += 1
                tag_counts.update(ex["ner_tags"])

                if state["written"] % 5000 == 0:
                    conn.commit()

    conn.commit()
    return state

_LIBRUSEC_FILES = [
    f"hf://datasets/IlyaGusev/librusec/{i:02d}.jsonl.zst" for i in range(20)
]
ds_librusec = load_dataset(
    "json", data_files=_LIBRUSEC_FILES, split="train", streaming=True
)

seen_db_path = os.path.join(DATA_DIR, "_rare_punct_seen.sqlite")
if os.path.exists(seen_db_path):
    os.remove(seen_db_path)
conn = sqlite3.connect(seen_db_path)
conn.execute("PRAGMA journal_mode=WAL")
conn.execute("PRAGMA synchronous=NORMAL")
conn.execute("CREATE TABLE seen_chunks (hash BLOB PRIMARY KEY)")

rare_path = os.path.join(DATA_DIR, "rare_punct_fiction.json")
with open(rare_path, "w", encoding="utf-8") as f:
    f.write("[\n")
    state = mine_from_texts(
        (item["text"] for item in ds_librusec), "librusec", "Mining LibruSec", conn, f
    )
    f.write("\n]\n")

conn.close()
if os.path.exists(seen_db_path):
    os.remove(seen_db_path)

print("\nMining counts:")
for cat in _CATEGORIES:
    print(f"  {cat:10s}: {_counts[cat]:>6d}")

print(
    f"\nKept {state['written']} chunks, "
    f"dropped {state['dropped_only_abundant']} (only O/./,), "
    f"duplicates {state['duplicates']}, "
    f"label errors {state['errors']}"
)


# ── Label-distribution audit ─────────────────────────────────────────────────
total = sum(tag_counts.values())
print(f"\nLabel distribution in mined data ({total} tokens):")
for label, idx in sorted(LABEL_MAP.items(), key=lambda x: -tag_counts.get(x[1], 0)):
    c = tag_counts.get(idx, 0)
    pct = 100 * c / total if total else 0
    print(f"  {idx:2d} {label!r:8s}: {c:>7d}  ({pct:.2f}%)")

print(f"\nWrote {state['written']} examples -> {rare_path}")

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Mining LibruSec: 0it [00:00, ?it/s]


Mining counts:
  !         : 11377694
  ?         : 11377740
  ;         : 7199316
  dialogue  : 11377625

Kept 36708418 chunks, dropped 12594 (only O/./,), duplicates 4611112, label errors 0

Label distribution in mined data (8494048530 tokens):
   0 'O'     : 6562838356  (77.26%)
   2 ','     : 902718132  (10.63%)
   1 '.'     : 507726551  (5.98%)
  23 '. -'   : 134116360  (1.58%)
   7 '-'     : 62620859  (0.74%)
  27 ', -'   : 56868070  (0.67%)
   8 '"'     : 49251581  (0.58%)
   4 '?'     : 43540369  (0.51%)
  26 '? -'   : 39073338  (0.46%)
   3 '!'     : 37544611  (0.44%)
   5 ':'     : 21277806  (0.25%)
  25 '! -'   : 19041484  (0.22%)
  15 '".'    : 12377565  (0.15%)
   6 ';'     : 11610130  (0.14%)
  14 '",'    : 8105053  (0.10%)
  10 ': "'   : 7438073  (0.09%)
  11 '. "'   : 6563862  (0.08%)
  17 '!"'    : 1842062  (0.02%)
  20 '", -'  : 1825839  (0.02%)
   9 ', "'   : 1748849  (0.02%)
  18 '" -'   : 1476014  (0.02%)
  16 '?"'    : 1316112  (0.02%)
  19 '- "'   :  959591  (0.

In [8]:
path_train_all = os.path.join(DATA_DIR, "train_all.json")
path_rare      = os.path.join(DATA_DIR, "rare_punct_5000.json")
path_merged    = os.path.join(DATA_DIR, "train_all_rare_5000.json")

with open(path_train_all, "r", encoding="utf-8") as f:
    base = json.load(f)
with open(path_rare, "r", encoding="utf-8") as f:
    rare = json.load(f)

merged = base + rare
random.Random(RANDOM_SEED).shuffle(merged)

with open(path_merged, "w", encoding="utf-8") as f:
    json.dump(merged, f, ensure_ascii=False)

print(f"Base train:  {len(base):>7d}")
print(f"Rare punct:  {len(rare):>7d}")
print(f"Merged:      {len(merged):>7d}")
print(f"Wrote {path_merged}")

del base, rare, merged
gc.collect()

Base train:    61243
Rare punct:     4467
Merged:        65710
Wrote /home/temari/god please no diploma/restore_punct/data/train_all_rare_5000.json


73